In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from datetime import datetime

from footy_track.object_detections import roboflow_labelling
from footy_track.object_detections.constants import DATA_DIR

## Configuration
Set your Roboflow workspace and project names below. The workspace name can be found in the URL of your Roboflow dashboard (e.g., `https://app.roboflow.com/your-workspace-name`).

In [ ]:
# Roboflow Configuration - PLEASE UPDATE THESE VALUES
WORKSPACE_NAME = "egroeg121"  # Replace with your workspace name/ID
PROJECT_NAME = "footy-track-broadcast-frame"   # Replace with your desired project name

## Initialize Handler
Instantiate the `RoboflowHandler`. This will connect to your workspace using the API key from your environment variables and then get or create the specified project.

In [ ]:
# Initialize the Roboflow Classification Handler
handler = roboflow_labelling.RoboflowClassificationHandler(
    workspace_name=WORKSPACE_NAME,
    project_name=PROJECT_NAME,
)
print(f"Using project: {handler.project.name}")
print(f"Project URL: https://app.roboflow.com/{WORKSPACE_NAME}/{PROJECT_NAME}")

## Upload Data
Run the upload process. The classification handler will perform the following steps:
1. Load local images from `DATA_DIR`.
2. Use the placeholder classifier to assign a `yes` or `no` label to each image (indicating if it's a broadcast frame).
3. Uploads each image to Roboflow with its classification label and tag.

In [ ]:
handler.upload_dir(
    # image_dir=Path("../data/arsenal_mancity_frames_example"),
    image_dir=DATA_DIR,
    sample_number=500,
    batch_name=f"arsenal_mancity_frames_initial_{datetime.now():%Y%m%d_%H%M%S}"
)

print("Upload complete.")

In [ ]:
import numpy as np

all_img_paths = list(DATA_DIR.glob("*.jpg"))
img_paths = all_img_paths

NUM_SAMPLES = 500
NUM_SAMPLES = None
if NUM_SAMPLES is not None:
    n = min(NUM_SAMPLES, len(all_img_paths))
    indices = np.linspace(0, len(all_img_paths), n, endpoint=False, dtype=int)
    img_paths = [all_img_paths[i] for i in indices]

In [ ]:
import fiftyone as fo
from tqdm import tqdm
# Use the handler to visualise using Voxel51

datapoints = []
for image_path in tqdm(img_paths):
    classification_result = handler.classifier.predict_from_path(image_path)    
    datapoints.append(classification_result.to_fiftyone_sample(image_path=image_path, key="prediction"))

dataset = fo.Dataset()
a = dataset.add_samples(datapoints)

In [ ]:
import fiftyone.brain as fob

# Compute and visualize embeddings with CLIP
brain_key = f"img_viz_{datetime.now():%Y%m%d_%H%M%S}"
fob.compute_visualization(
    dataset,
    model="clip-vit-base32-torch",
    brain_key=brain_key,
)

In [ ]:
session = fo.launch_app(dataset)

In [ ]:
files_to_upload = []

In [ ]:
selected_ids = session.selected

# Now map those to file paths
filepaths = [dataset[sid].filepath for sid in selected_ids]
files_to_upload.extend(filepaths)
files_to_upload = list(set(files_to_upload))
print(f"{len(files_to_upload)=}")

In [ ]:
files_to_upload = [Path(p) for p in files_to_upload]

In [ ]:
from footy_track.classifier import RoboflowClassifier


handler.classifier = RoboflowClassifier(
            workspace_name=handler.workspace.name, project_name=handler.project_name, api_key=handler.api_key,version=4
        )

In [ ]:
handler.upload_images(
    image_paths=files_to_upload,
    batch_name=f"arsenal_mancity_frames_bad_performing_{datetime.now():%Y%m%d_%H%M%S}"
)
